### Steps that has been done in the notebook
1. Read bronze ciruits data
2. Keep the only required columns for analytics, will drop the url column
3. Standardize the column name with snake case (ex: circuitID --> circuit_id)
4. Renaming Columns to make it more meaningful
5. Buisness key validation: Filter out rows where circuit_id is null
6. Remove duplicate records
7. Transform values of columns to title case (ex: circuit_name)
8. Write the data to silver circuits data

In [0]:
%run ../environment_config

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.circuits"
silver_table = f"{catalog}.{silver_schema}.circuits"

In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_df = spark.read.table(bronze_table)

In [0]:
# SELECTING COLUMNS

circuits_selected_df = circuits_df.select( 
        F.col("circuitId"), F.col("circuitName"),
        F.col("lat"), F.col("long"), F.col("locality"),
        F.col("country"), F.col("ingestion_timestamp"), F.col("souce_file"), F.col("source_path")
)

# ANOTHER WAY OF SELECTING COLUMN (SINCE I WANT TO DROP ONLY ONE COLUMN)
# circuits_selected_df = circuits_df.select("*").drop(F.col)

In [0]:
circuits_renamed_df = circuits_selected_df.withColumnsRenamed({
                                        "circuitId": "circuit_id",
                                        "circuitName": "circuit_name",
                                        "lat": "latitude",
                                        "long": "longitude"
                                    })

In [0]:
## KEEPING ONLY UNIQUE ROWS
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
display(circuits_distinct_df)

In [0]:
## STANDARDIZING COLUMN DATA
circuits_final_df = (circuits_distinct_df
                              .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
                              .withColumn("locality", F.initcap(F.col("locality")))
                    )

In [0]:
display(circuits_final_df)

In [0]:
## WRITING DATA TO TABLE
(
    circuits_final_df.write
                     .format("delta")
                     .mode("overwrite")
                     .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))